# Crossfire 四分类微调（Google Colab / T4）

从上往下依次运行每一格。训练**不需要** API key。

开始前：菜单 **代码执行程序 → 更改运行时类型 → T4 GPU**。

结果会写到你的 Google Drive：`MyDrive/crossfire-runs/`。每个种子训完就同步一次，中途断线也不至于全丢。

## 1. 检查有没有 GPU

打印 `nvidia-smi`。如果报错或没有 GPU，说明运行时还是 CPU——去改成 T4，然后**停止并重新连接**，再从这一格重跑。

大概几秒钟。

In [ ]:
import subprocess, sys
r = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(r.stdout or r.stderr)
if r.returncode != 0:
    print("\n❌ 没有 GPU。请：代码执行程序 → 更改运行时类型 → T4 GPU，然后重新连接运行时。")
    sys.exit("No GPU")
print("✅ 看到 GPU 信息就可以继续下一格。")

## 2. 挂载 Google Drive

把结果存进云盘，避免 Colab 会话断开后文件消失。首次会弹出授权窗口，点允许即可。

会在 Drive 里建 `crossfire-runs/`。大约半分钟（含授权）。

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")
RUNS = Path("/content/drive/MyDrive/crossfire-runs")
RUNS.mkdir(parents=True, exist_ok=True)
print("结果目录：", RUNS)

## 3. 克隆仓库并安装训练依赖

用 `uv sync --group train`，版本和本地 `pyproject.toml` 一致（torch / transformers 等）。

首次大约 3–8 分钟（下载 torch 较大）。

In [ ]:
%%bash
set -euo pipefail
cd /content
rm -rf Crossfire
git clone --depth 1 https://github.com/FishySean/Crossfire.git
cd Crossfire
curl -LsSf https://astral.sh/uv/install.sh | sh
export PATH="$HOME/.local/bin:$PATH"
uv sync --group train
wc -l data/pairs.jsonl

## 4. 自检：CUDA、显卡型号、是否支持 bf16

`torch.cuda.is_available()` 必须是 True。T4 通常 **不支持 bf16**，脚本会自动改用 **fp16 + GradScaler**（参数仍是 fp32，安全）。

这一格用 `%%bash` + heredoc（不要用 `!` 包 heredoc，IPython 只会执行第一行）。

几秒钟。

In [ ]:
%%bash
cd /content/Crossfire
export PATH="$HOME/.local/bin:$PATH"
# 必须拉最新代码：bf16 按算力版本判断，不再信 is_bf16_supported 的默认模拟
git pull --ff-only origin main
uv run python - <<'PY'
import torch
from training.amp import choose_amp, describe_device

assert torch.cuda.is_available(), "CUDA 不可用：请换成 T4 GPU 运行时"
device = torch.device("cuda")
cap = torch.cuda.get_device_capability(device)
amp = choose_amp(device)
print("显卡：", describe_device(device))
print("算力版本：", f"{cap[0]}.{cap[1]}")
print("is_bf16_supported() 默认：", torch.cuda.is_bf16_supported())  # T4 可能仍是 True（模拟）
print("将使用：", amp.name)
print("原因：", amp.reason)
expected = "fp16+scaler" if cap[0] < 8 else "bf16"
assert amp.name == expected, f"算力 {cap} 应选 {expected}，实际 {amp.name}"
PY

## 5. GPU 冒烟（--smoke）

只跑 1 个种子、很少的 batch，确认训练循环、存盘、画图都通。结束会打印**峰值显存**。

大约 1–3 分钟。

In [ ]:
%%bash
set -euo pipefail
export PATH="$HOME/.local/bin:$PATH"
cd /content/Crossfire
SMOKE_MIRROR="/content/drive/MyDrive/crossfire-runs/smoke"
mkdir -p "$SMOKE_MIRROR"
uv run python -m training.train --smoke --mirror-dir "$SMOKE_MIRROR"

## 6. 正式训练（三个种子）

种子 42 / 43 / 44。早停看验证集加权损失。每个种子结束后立刻同步到 Drive。

T4 上大约 **15–25 分钟**。可以去喝水，但别关标签页太久（闲置可能断连；有 Drive 同步可兜底）。

In [ ]:
%%bash
set -euo pipefail
export PATH="$HOME/.local/bin:$PATH"
cd /content/Crossfire
stamp=$(date -u +%Y%m%d-%H%M%S)
MIRROR="/content/drive/MyDrive/crossfire-runs/run-${stamp}"
mkdir -p "$MIRROR"
echo "Drive 同步目录：$MIRROR"
uv run python -m training.train --mirror-dir "$MIRROR"

## 7. 查看打包结果

打包里有：fp16 最终模型、三个种子的报告和曲线、summary。**没有**其它种子的大 checkpoint，也没有优化器状态。

在 Drive 的 `crossfire-runs/run-.../package/` 里下载即可。

In [ ]:
from pathlib import Path
import json

mirror = Path("/content/drive/MyDrive/crossfire-runs")
runs = sorted(mirror.glob("run-*"), key=lambda p: p.stat().st_mtime)
assert runs, "还没有 run-* 目录，请先跑完正式训练那一格"
latest = runs[-1]
pack = latest / "package"
print("最新运行：", latest)
print("打包目录：", pack)
if (pack / "pack_meta.json").exists():
    print(json.dumps(json.loads((pack / "pack_meta.json").read_text()), indent=2, ensure_ascii=False))
if (latest / "summary.json").exists():
    s = json.loads((latest / "summary.json").read_text())
    print("入选种子：", s.get("selected_seed_by_val_loss"))
    print("三种子测试宏 F1：", s.get("test_macro_f1_by_seed"))
    print("汇总：", s.get("test_macro_f1_summary"))

## 做完了

从 Google Drive 下载 `package/` 文件夹。Colab 可以断开；**没有**像 RunPod 那样的磁盘持续计费。

若还开着其它付费云主机（算力自由 / RunPod），请登录控制台确认已 **删除 / Terminate**，避免继续扣费。